# Risco de atraso em pedidos da Olist

## Desafio de negócio

Identificar, logo após a aprovação do pagamento, quais pedidos possuem maior risco de serem entregues depois do prazo prometido. A análise usa somente dados disponíveis nesse instante; a data real de entrega é usada exclusivamente para construir o alvo.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", palette="Blues_r")

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data" / "raw"
DATA_DIR

## 1. Carregamento, alvo e integração

O alvo é `is_late = 1` quando a entrega ao cliente ocorre após a data prometida. O recorte contém apenas pedidos entregues com as datas necessárias e com aprovação registrada. A tabela de itens é agregada antes do *merge* para preservar uma linha por pedido.

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv", parse_dates=date_columns)
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")

delivered_orders = orders.loc[
    orders["order_status"].eq("delivered")
    & orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
    & orders["order_approved_at"].notna()
].copy()
delivered_orders["is_late"] = (
    delivered_orders["order_delivered_customer_date"]
    > delivered_orders["order_estimated_delivery_date"]
).astype("int8")

items_agg = items.groupby("order_id", as_index=False).agg(
    item_count=("order_item_id", "count"),
    seller_count=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
)

orders_analysis = (
    delivered_orders
    .merge(items_agg, on="order_id", how="left", validate="one_to_one")
    .merge(
        customers[["customer_id", "customer_state", "customer_city"]],
        on="customer_id", how="left", validate="many_to_one",
    )
)
assert orders_analysis["order_id"].is_unique

orders_analysis["promised_days"] = (
    orders_analysis["order_estimated_delivery_date"]
    - orders_analysis["order_approved_at"]
).dt.total_seconds().div(86_400)
orders_analysis["purchase_month"] = orders_analysis["order_purchase_timestamp"].dt.month
orders_analysis.shape

## Pergunta 1: Qual é a frequência de pedidos atrasados?

**Hipótese:** os atrasos são minoritários, produzindo desbalanceamento entre as classes.

In [ ]:
late_summary = orders_analysis["is_late"].value_counts().rename(index={0: "No prazo", 1: "Atrasado"}).to_frame("pedidos")
late_summary["percentual"] = late_summary["pedidos"] / late_summary["pedidos"].sum() * 100
display(late_summary.round(2))

ax = late_summary["percentual"].plot(kind="bar", ylabel="Pedidos (%)", rot=0)
ax.bar_label(ax.containers[0], fmt="%.1f%%")
plt.title("Distribuição do alvo de atraso")
plt.show()

**Conclusão da pergunta 1:** dos **96.456** pedidos analisados, **7.826 (8,11%)** atrasaram e 88.630 (91,89%) foram entregues no prazo. A hipótese foi apoiada: as classes são desbalanceadas. Um modelo que sempre previsse “não atrasará” teria 91,89% de acurácia, mas não identificaria nenhum atraso; por isso acurácia sozinha não é uma métrica adequada. Métricas como *recall*, precisão, PR-AUC e custo operacional devem complementar a avaliação.

## Pergunta 2: Pedidos com prazos menores atrasam mais?

**Hipótese:** prazos muito curtos deixam menos margem para problemas logísticos e podem elevar o risco.

In [ ]:
bins = [0, 7, 14, 21, 30, np.inf]
labels = ["até 7", "8–14", "15–21", "22–30", "mais de 30"]
orders_analysis["promised_days_band"] = pd.cut(
    orders_analysis["promised_days"], bins=bins, labels=labels, include_lowest=True
)
deadline_summary = orders_analysis.groupby("promised_days_band", observed=False).agg(
    pedidos=("is_late", "size"),
    taxa_atraso=("is_late", "mean"),
)
deadline_summary["taxa_atraso"] *= 100
display(deadline_summary.round(2))

ax = deadline_summary["taxa_atraso"].plot(kind="bar", ylabel="Atrasos (%)", rot=0)
ax.bar_label(ax.containers[0], fmt="%.1f%%")
plt.title("Taxa de atraso por prazo prometido")
plt.show()

**Conclusão da pergunta 2:** a taxa de atraso é mais alta para prazo de até 7 dias (**28,34%**, 1.637 pedidos). Nas demais faixas ela varia de 5,38% a 9,07%: 8–14 dias (6,76%), 15–21 (9,07%), 22–30 (8,32%) e mais de 30 (5,38%). A hipótese é fortemente apoiada para o prazo mais curto. Essa primeira faixa tem menos observações que as demais, mas ainda conta com mais de 1,6 mil pedidos; merece atenção e possivelmente uma feature não linear no modelo.

## Pergunta 3: Pedidos maiores ou mais complexos atrasam mais?

**Hipótese:** pedidos com mais itens ou vendedores apresentam maior complexidade logística e maior risco.

In [ ]:
complexity_summary = orders_analysis.groupby("is_late").agg(
    pedidos=("order_id", "size"),
    mediana_itens=("item_count", "median"),
    media_itens=("item_count", "mean"),
    mediana_vendedores=("seller_count", "median"),
    media_vendedores=("seller_count", "mean"),
).rename(index={0: "No prazo", 1: "Atrasado"})
display(complexity_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=orders_analysis, x="is_late", y="item_count", ax=axes[0])
sns.boxplot(data=orders_analysis, x="is_late", y="seller_count", ax=axes[1])
for ax, title in zip(axes, ["Itens por pedido", "Vendedores por pedido"]):
    ax.set_xticks([0, 1], ["No prazo", "Atrasado"])
    ax.set_title(title)
plt.tight_layout()
plt.show()

**Conclusão da pergunta 3:** a hipótese não foi apoiada neste recorte. A mediana é de **1 item** e **1 vendedor** tanto para pedidos no prazo quanto para atrasados; os três quartis também são iguais a 1 nos dois grupos. As médias são até levemente menores nos atrasados (1,11 itens e 1,00 vendedor) do que nos pedidos no prazo (1,15 e 1,02). Os boxplots devem apresentar forte sobreposição, portanto essas duas variáveis isoladas provavelmente têm baixo poder preditivo; podem ser mantidas para o modelo testar interações com estado, prazo e frete.

## Pergunta 4: O risco varia geograficamente?

**Hipótese:** alguns estados apresentam maior risco devido à distância ou à infraestrutura logística. São exibidos somente estados com pelo menos 100 pedidos.

In [ ]:
state_summary = orders_analysis.groupby("customer_state").agg(
    pedidos=("is_late", "size"), taxa_atraso=("is_late", "mean")
).query("pedidos >= 100").sort_values("taxa_atraso", ascending=False)
state_summary["taxa_atraso"] *= 100
display(state_summary.round(2))

fig, ax = plt.subplots(figsize=(12, 4))
state_summary["taxa_atraso"].plot(kind="bar", ax=ax, ylabel="Atrasos (%)")
ax.bar_label(ax.containers[0], fmt="%.1f%%", fontsize=8)
plt.title("Taxa de atraso por estado do cliente")
plt.show()

**Conclusão da pergunta 4:** há variação geográfica relevante. Entre estados com ao menos 100 pedidos, AL (23,93%), MA (19,69%), PI (15,97%) e CE (15,34%) têm as maiores taxas; RO (2,88%), AM (4,14%), PR (5,00%) e MG (5,61%) têm as menores. Assim, o estado do cliente pode ser útil como feature. Porém ele provavelmente representa distância entre cliente e vendedor, malha de transporte, cobertura de transportadoras, densidade urbana, prazo prometido e composição de produtos — não uma causa isolada.

## Pergunta 5: O momento da compra está associado ao atraso?

**Hipótese:** determinados meses enfrentam maior pressão logística e apresentam maior taxa de atraso.

In [ ]:
month_summary = orders_analysis.groupby("purchase_month").agg(
    pedidos=("is_late", "size"), taxa_atraso=("is_late", "mean")
)
month_summary["taxa_atraso"] *= 100
display(month_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
month_summary["taxa_atraso"].plot(kind="bar", ax=axes[0], ylabel="Atrasos (%)", rot=0)
month_summary["pedidos"].plot(kind="bar", ax=axes[1], ylabel="Pedidos", rot=0)
axes[0].set_title("Taxa de atraso por mês da compra")
axes[1].set_title("Volume de pedidos por mês da compra")
plt.tight_layout()
plt.show()

**Conclusão da pergunta 5:** há indícios de sazonalidade. Março (17,15%), novembro (14,31%) e fevereiro (13,43%) concentram as maiores taxas; junho tem a menor (2,21%). O padrão não é explicado apenas pelo volume: novembro tem 7.288 pedidos, abaixo de maio, junho, julho e agosto, enquanto março combina taxa alta com 9.549 pedidos. Como o recorte abrange apenas um período histórico, mês deve ser usado com cautela e validado temporalmente; também pode capturar mudanças de operação, não só sazonalidade.

## Síntese para modelagem

As evidências mais promissoras são o prazo prometido, a localização do cliente e o mês da compra. Contagens de itens e vendedores não mostraram diferença marginal relevante. O próximo passo é criar um conjunto de treino respeitando o tempo, codificar as variáveis disponíveis após a aprovação e avaliar o modelo com métricas sensíveis à classe minoritária e ao custo de intervenção.